In [1]:
!pip install -q geemap

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 42.6 MB/s eta 0:00:00


In [3]:
import ee
ee.Authenticate(auth_mode='notebook')
ee.Initialize(project='glofwatch-india')
print("Earth Engine connected")

To authorize access needed by Earth Engine, open the following URL in a web browser and follow the instructions. If the web browser does not start automatically, please manually browse the URL below.

    https://code.earthengine.google.com/client-auth?scopes=https%3A//www.googleapis.com/auth/earthengine%20https%3A//www.googleapis.com/auth/cloud-platform%20https%3A//www.googleapis.com/auth/drive%20https%3A//www.googleapis.com/auth/devstorage.full_control&request_id=v00ur0zOm68pr8lOL9TeLrA1DWrCV-IqG8G3-EGbnfE&tc=A7FxzN3DzQZC6FvshY8MRm-DNAc3t_mc4GWKl26NPd4&cc=tqmvKwww_I4sVMxFOmLJFBat98MgJbMVJoi8AoZ5eZU

The authorization workflow will generate a code, which you should paste in the box below.
Enter verification code: 4/1AXlqoi53X5_-xqqV0fVnOG53QwqflYD9uTgK6TV__qFv01o6CmVRwhoO1ss

Successfully saved authorization token.
Earth Engine connected


In [6]:
import geemap

lake = ee.Geometry.Point([88.18, 27.91])
img = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
       .filterBounds(lake)
       .filterDate('2025-09-01', '2025-10-31')
       .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
       .median())

m = geemap.Map(center=[27.91, 88.18], zoom=13, basemap='Esri.WorldImagery')
m.addLayer(img, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2')
m

Map(center=[27.91, 88.18], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [9]:
aoi = ee.Geometry.Rectangle([88.10, 27.85, 88.25, 28.00])  # South Lhonak area

def mask_s2(img):
    scl = img.select('SCL')
    # drop cloud shadow (3), medium/high cloud (8, 9), cirrus (10)
    good = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))
    return img.updateMask(good)

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate('2025-09-01', '2025-10-31')
      .map(mask_s2)
      .median()
      .clip(aoi))

m = geemap.Map(basemap='Esri.WorldImagery')
m.centerObject(aoi, 13)
m.addLayer(s2, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2 RGB')
m

Map(center=[27.925002966337622, 88.17499999999927], controls=(WidgetControl(options=['position', 'transparent_…

In [10]:
mndwi = s2.normalizedDifference(['B3', 'B11']).rename('MNDWI')
water = mndwi.gt(0).selfMask()

m.addLayer(mndwi, {'min': -1, 'max': 1, 'palette': ['brown', 'white', 'blue']}, 'MNDWI', False)
m.addLayer(water, {'palette': ['cyan']}, 'Water mask')
m

Map(bottom=7033063.0, center=[27.918074915705876, 88.19293582049285], controls=(WidgetControl(options=['positi…

In [11]:
area_img = water.multiply(ee.Image.pixelArea())
area = area_img.reduceRegion(
    reducer=ee.Reducer.sum(), geometry=aoi, scale=20, maxPixels=1e9
).getInfo()
print(f"Water area in AOI: {area['MNDWI']/1e6:.2f} km²")

Water area in AOI: 226.71 km²
